# FOD Tespiti - YOLO11 Egitimi (Colab)
Calisma zamani > Calisma zamani turunu degistir > **T4 GPU** secin.
Once yerelde `data/yolo.zip` olusturup Google Drive'a `MyDrive/fod/yolo.zip` olarak yukleyin.

In [ ]:
!pip -q install ultralytics
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/data && unzip -q -o /content/drive/MyDrive/fod/yolo.zip -d /content/data
!ls /content/data/yolo

In [ ]:
# data.yaml icindeki path'i Colab yoluna sabitle
import re
p = '/content/data/yolo/data.yaml'
t = open(p).read()
t = re.sub(r'^path:.*$', 'path: /content/data/yolo', t, flags=re.M)
open(p, 'w').write(t)
print(t)

In [ ]:
from ultralytics import YOLO
MODEL = 'yolo11s.pt'  # once 'yolo11n.pt' ile hizli deneme yapilabilir
model = YOLO(MODEL)
model.train(
    data='/content/data/yolo/data.yaml',
    epochs=50, imgsz=320, batch=64, patience=15,
    project='/content/drive/MyDrive/fod/runs', name=MODEL.replace('.pt',''),
    mosaic=1.0, hsv_h=0.015, hsv_s=0.6, hsv_v=0.4, degrees=10, fliplr=0.5, flipud=0.2,
)

In [ ]:
# Recall odakli degerlendirme (dusuk conf)
best = YOLO(f'/content/drive/MyDrive/fod/runs/{MODEL.replace(".pt","")}/weights/best.pt')
m = best.val(data='/content/data/yolo/data.yaml', conf=0.1)
print('mAP50:', m.box.map50, 'Precision:', m.box.mp, 'Recall:', m.box.mr)

In [ ]:
# ONNX disa aktarim (yerel CPU/DirectML cikarimi icin)
best.export(format='onnx', imgsz=320, simplify=True, opset=12)